Entity

In [3]:
from dataclasses import dataclass
from pathlib import Path
import os
@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    architecture: str
    pretrained_weights: str
    num_classes: int

In [4]:
os.getcwd()

'd:\\Personal_projects\\Pyhton_proj\\AI-Food-Recognition-Nutrition-Assistant\\research'

In [5]:
os.chdir("..")

In [6]:
%pwd

'd:\\Personal_projects\\Pyhton_proj\\AI-Food-Recognition-Nutrition-Assistant'

Config Manager

In [7]:
from AI_Food_Recognition_Nutrition_Assistant.constants import *
from AI_Food_Recognition_Nutrition_Assistant.utils.common import read_yaml,create_directories

In [8]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        p = self.params
        create_directories([config.root_dir])
        return PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            architecture=p.model.architecture,
            pretrained_weights=p.model.pretrained_weights,
            num_classes=p.training.num_classes,
        )

Prep Base model Component

In [9]:
import zipfile
import urllib.request as request
from AI_Food_Recognition_Nutrition_Assistant import logger
from AI_Food_Recognition_Nutrition_Assistant.utils.common import get_size

In [ ]:
import torch.nn as nn
import torch
from torchvision import models

class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    def get_base_model(self) -> nn.Module:
        logger.info(f"Loading {self.config.architecture} with weights={self.config.pretrained_weights}")
        model = models.convnext_tiny(weights=self.config.pretrained_weights)
        create_directories([self.config.root_dir])
        torch.save(model.state_dict(),self.config.base_model_path) 
        return model

    def update_model_head(self, model: nn.Module) -> nn.Module:
        in_features = model.classifier[2].in_features# type: ignore[index]
        model.classifier[2] = nn.Linear(in_features, self.config.num_classes)# type: ignore[index]
        logger.info(f"Updated classifier head: {in_features} -> {self.config.num_classes} classes")
        torch.save(model.state_dict(), self.config.updated_base_model_path)
        return model

    def prepare(self) -> nn.Module:
        model = self.get_base_model()
        model = self.update_model_head(model)
        return model

Pipeline

In [18]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    preparer = PrepareBaseModel(prepare_base_model_config)
    model = preparer.prepare()
except Exception as e:
    raise e

[2026-04-18 16:36:01,169: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-18 16:36:01,173: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-18 16:36:01,175: INFO: common: created directory at: artifacts]
[2026-04-18 16:36:01,176: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-04-18 16:36:01,177: INFO: 3154001565: Loading convnext_tiny with weights=IMAGENET1K_V1]
[2026-04-18 16:36:01,588: INFO: common: created directory at: artifacts\prepare_base_model]
[2026-04-18 16:36:01,672: INFO: 3154001565: Updated classifier head: 768 -> 101 classes]
